# Actividad 4: AdaBoost Implementation
## Héctor Arturo Hernández Escalante
### Profesor: Dr. Victor Uc
### Maestría en Ciencias de la Computación
### Segundo Semestre
### Universidad Autónoma de Yucatán
### 20 de Mayo 2026

### 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


### 2. Data Loading Utility

In [ ]:
def load_data(file_path):
    """
    Loads data from a text file, separates features and labels, and ensures labels are -1 and 1.

    Args:
        file_path (str): Path to the dataset file.

    Returns:
        tuple: (X, y) where X is a numpy array of features and y is a numpy array of labels.
    """
    data = np.loadtxt(file_path)
    X = data[:, :-1]
    y = data[:, -1]
    # Convert labels 0 to -1 if present
    y[y == 0] = -1
    return X, y


### 3. AdaBoost Implementation
#### 3.1 Decision Stump

In [ ]:
class DecisionStump:
    """
    A Decision Stump is a simple one-level decision tree used as a weak classifier in AdaBoost.
    It splits data based on a single feature and a threshold.
    """
    def __init__(self):
        """
        Initializes the Decision Stump with default values.
        """
        self.feature_idx = None
        self.threshold = None
        self.polarity = 1
        self.alpha = None

    def predict(self, X):
        """
        Predicts labels for a given set of features.

        Args:
            X (numpy.ndarray): Feature matrix.

        Returns:
            numpy.ndarray: Predicted labels (-1 or 1).
        """
        n_samples = X.shape[0]
        X_column = X[:, self.feature_idx]
        predictions = np.ones(n_samples)
        if self.polarity == 1:
            predictions[X_column < self.threshold] = -1
        else:
            predictions[X_column >= self.threshold] = -1
        return predictions


#### 3.2 AdaBoost Class

In [ ]:
class AdaBoost:
    """
    Implementation of the AdaBoost (Adaptive Boosting) algorithm.
    """
    def __init__(self, n_clf=50):
        """
        Initializes the AdaBoost model.

        Args:
            n_clf (int): Number of weak classifiers to use. Defaults to 50.
        """
        self.n_clf = n_clf
        self.clfs = []

    def fit(self, X, y):
        """
        Trains the AdaBoost model using the provided training data.

        Args:
            X (numpy.ndarray): Feature matrix of shape (n_samples, n_features).
            y (numpy.ndarray): Target labels of shape (n_samples,).
        """
        n_samples, n_features = X.shape
        # Initialize weights to 1/N
        w = np.full(n_samples, (1 / n_samples))

        self.clfs = []

        for _ in range(self.n_clf):
            clf = DecisionStump()
            min_error = float('inf')

            # Greedy search for best threshold and feature
            for feature_i in range(n_features):
                X_column = X[:, feature_i]
                thresholds = np.unique(X_column)

                for threshold in thresholds:
                    # predict with polarity 1
                    p = 1
                    predictions = np.ones(n_samples)
                    predictions[X_column < threshold] = -1

                    # Error = sum of weights of misclassified samples
                    error = sum(w[y != predictions])

                    if error > 0.5:
                        error = 1 - error
                        p = -1

                    if error < min_error:
                        clf.polarity = p
                        clf.threshold = threshold
                        clf.feature_idx = feature_i
                        min_error = error

            # Calculate alpha
            # Add a small epsilon to avoid division by zero
            EPS = 1e-10
            clf.alpha = 0.5 * np.log((1.0 - min_error + EPS) / (min_error + EPS))

            # Update weights
            predictions = clf.predict(X)
            w *= np.exp(-clf.alpha * y * predictions)
            # Normalize to 1
            w /= np.sum(w)

            self.clfs.append(clf)

    def predict(self, X):
        """
        Predicts labels for the given feature matrix using the ensemble of weak classifiers.

        Args:
            X (numpy.ndarray): Feature matrix of shape (n_samples, n_features).

        Returns:
            numpy.ndarray: Predicted labels (-1 or 1).
        """
        clf_preds = [clf.alpha * clf.predict(X) for clf in self.clfs]
        y_pred = np.sum(clf_preds, axis=0)
        y_pred = np.sign(y_pred)
        return y_pred


### 4. Visualization Utility

In [ ]:
def plot_decision_boundary(clf, X, y, title):
    """
    Plots the decision boundary of a classifier along with the data points.

    Args:
        clf: The trained classifier with a predict method.
        X (numpy.ndarray): Feature matrix.
        y (numpy.ndarray): Target labels.
        title (str): Title for the plot.
    """
    h = .02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    plt.contourf(xx, yy, Z, cmap=plt.cm.Paired, alpha=0.8)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.Paired, edgecolors='k')
    plt.title(title)
    plt.show()


### 5. Execution and Results

In [ ]:
print("Processing dataCircle.txt")
X2, y2 = load_data("../datasets/dataCircle.txt")
ada2 = AdaBoost(n_clf=20)
ada2.fit(X2, y2)
y_pred2 = ada2.predict(X2)
accuracy2 = np.mean(y2 == y_pred2)
print(f"Accuracy on dataCircle.txt: {accuracy2 * 100:.2f}%")
plot_decision_boundary(ada2, X2, y2, "AdaBoost on dataCircle.txt")
